In [8]:
import os
import json
from pathlib import Path
from urllib.parse import urlparse

import httpx
from tqdm import tqdm
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import Dataset, load_dataset


def init_login():
    load_dotenv()
    login(os.getenv("HUGGINGFACE_TOKEN"))
    print("Login successful")

In [3]:
ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"

In [10]:
def download_image(image_url, save_path, timeout=10):
    """
    Download an image from a URL and save it to the specified path using httpx.
    
    Args:
        image_url (str): URL of the image to download
        save_path (str): Local path where to save the image (including filename)
        timeout (int): Request timeout in seconds
    
    Returns:
        bool: True if download successful, False otherwise
    """
    try:
        with httpx.Client() as client:
            response = client.get(image_url, timeout=timeout)
            response.raise_for_status()  # Raise an exception for bad status codes
            
            # Create directory if it doesn't exist
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            
            # Write the image to file
            with open(save_path, 'wb') as file:
                file.write(response.content)
        
        print(f"✓ Downloaded: {save_path}")
        return True
        
    except httpx.RequestError as e:
        print(f"✗ Failed to download {image_url}: {e}")
        return False
    except Exception as e:
        print(f"✗ Error saving image: {e}")
        return False

def read_json(file_path):
    """Read and return data from a JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"✗ File not found: {file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON in {file_path}: {e}")
        return None

def write_json(data, file_path, indent=4):
    """Write data to a JSON file."""
    try:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=indent, ensure_ascii=False)
        print(f"✓ Written JSON to: {file_path}")
        return True
    except Exception as e:
        print(f"✗ Error writing JSON: {e}")
        return False

# # Example usage:
# if __name__ == "__main__":
#     # Single image download
#     image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/example.jpg/250px-example.jpg"
#     save_path = "images/example.jpg"
#     download_image(image_url, save_path)
    
#     # Batch download from your JSON data
#     def download_images_from_json(json_file, images_dir="images"):
#         """Download all images from your scraped JSON data"""
#         data = read_json(json_file)
#         if data is None:
#             return
        
#         os.makedirs(images_dir, exist_ok=True)
        
#         for item in data:
#             title = item.get('title', 'unknown')
#             image_url = item.get('image_url', '')
            
#             if image_url:
#                 # Create safe filename from title
#                 safe_filename = "".join(c for c in title if c.isalnum() or c in (' ', '-', '_')).rstrip()
#                 filename = f"{safe_filename}.jpg"
#                 save_path = os.path.join(images_dir, filename)
                
#                 download_image(image_url, save_path)

    # For your German politicians data:
    # download_images_from_json('crawling_wiki/german_politicians_detail_01.json')
    
    # Example JSON operations
    # sample_data = [{'title': 'Test', 'url': 'example.com'}]
    # write_json(sample_data, 'test_output.json')

## Old days 

In [7]:
DATASET_DIR = DATA_DIR / "raw" / "wikipedia_famous_people" / "famous_people.json"

In [9]:
ds = pd.read_json(str(DATASET_DIR))
ds.head()

,name,url,summary,image_url,categories,timestamp
0,Sacheus !Gonteb,https://en.wikipedia.org/wiki/Sacheus_!Gonteb,Rear Admiral Sacheus Randy !Gonteb is a Namibi...,NaN,"[Articles with short description, Living peopl...",2025-08-02 04:53:49.485904
1,1.Cuz,https://en.wikipedia.org/wiki/1.Cuz,"Abas Abdikarim Bakar, better known as 1.Cuz (b...",NaN,"[1997 births, 21st-century male rappers, Artic...",2025-08-02 04:53:51.707057
2,1da Banton,https://en.wikipedia.org/wiki/1da_Banton,"Godson Ominibie Epelle, professionally known a...",NaN,"[1994 births, 21st-century Nigerian musicians,...",2025-08-02 04:53:54.208175
3,1nonly,https://en.wikipedia.org/wiki/1nonly,"Nathan Scott Fuller (born April 6, 2004), bett...",NaN,"[2004 births, 21st-century American male rappe...",2025-08-02 04:53:56.621336
4,1ucid,https://en.wikipedia.org/wiki/1ucid,"Kwadwo Bedihene, known professionally as 1ucid...",NaN,"[20th-century births, Afrobeats musicians, All...",2025-08-02 04:53:59.252879


In [10]:
ds = ds.drop(columns=["image_url"])
ds.head()

,name,url,summary,categories,timestamp
0,Sacheus !Gonteb,https://en.wikipedia.org/wiki/Sacheus_!Gonteb,Rear Admiral Sacheus Randy !Gonteb is a Namibi...,"[Articles with short description, Living peopl...",2025-08-02 04:53:49.485904
1,1.Cuz,https://en.wikipedia.org/wiki/1.Cuz,"Abas Abdikarim Bakar, better known as 1.Cuz (b...","[1997 births, 21st-century male rappers, Artic...",2025-08-02 04:53:51.707057
2,1da Banton,https://en.wikipedia.org/wiki/1da_Banton,"Godson Ominibie Epelle, professionally known a...","[1994 births, 21st-century Nigerian musicians,...",2025-08-02 04:53:54.208175
3,1nonly,https://en.wikipedia.org/wiki/1nonly,"Nathan Scott Fuller (born April 6, 2004), bett...","[2004 births, 21st-century American male rappe...",2025-08-02 04:53:56.621336
4,1ucid,https://en.wikipedia.org/wiki/1ucid,"Kwadwo Bedihene, known professionally as 1ucid...","[20th-century births, Afrobeats musicians, All...",2025-08-02 04:53:59.252879


In [12]:
hf_ds = Dataset.from_pandas(df=ds)
hf_ds

Dataset({
    features: ['name', 'url', 'summary', 'categories', 'timestamp'],
    num_rows: 1300
})

In [14]:
data_id = "minhleduc/wiki_famous_person_00"

hf_ds.push_to_hub(data_id, commit_message="Initial commit")

Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.07s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/minhleduc/wiki_famous_person_00/commit/1fd688c7abaa1f6cbf506a834c2a75cbacca7549', commit_message='Initial commit', commit_description='', oid='1fd688c7abaa1f6cbf506a834c2a75cbacca7549', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/minhleduc/wiki_famous_person_00', endpoint='https://huggingface.co', repo_type='dataset', repo_id='minhleduc/wiki_famous_person_00'), pr_revision=None, pr_num=None)

## 02.10.2025

In [6]:
datafile = str(DATA_DIR / "german" / "german_artist_detail.json")
json_data = read_json(datafile)

In [12]:
count = 0

for data in json_data:
    image_url = data['image_urls']
    if not image_url:
        continue
    image_url = image_url[0]
    name = data['title'].replace(" ", "_")
    download_image(image_url, f"images/{name}.jpg")


✓ Downloaded: images/Heinrich_Aldegrever.jpg
✓ Downloaded: images/Elisabeth_von_Adlerflycht.jpg
✓ Downloaded: images/Albrecht_Altdorfer.jpg
✓ Downloaded: images/Jean_Arp.jpg
✓ Downloaded: images/Asam_brothers.jpg
✓ Downloaded: images/Cosmas_Damian_Asam.jpg
✓ Downloaded: images/Egid_Quirin_Asam.jpg
✓ Downloaded: images/Isidor_Ascheim.jpg
✓ Downloaded: images/Jim_Avignon.jpg
✓ Downloaded: images/Johannes_Baader.jpg
✓ Downloaded: images/Caroline_Bardua.jpg
✓ Downloaded: images/Johann_Wolfgang_Baumgartner.jpg
✓ Downloaded: images/Barthel_Beham.jpg
✓ Downloaded: images/Hans_Bellmer.jpg
✓ Downloaded: images/Ella_Bergmann-Michel.jpg
✓ Downloaded: images/Joseph_Beuys.jpg
✓ Downloaded: images/Anna_and_Bernhard_Blume.jpg
✓ Downloaded: images/Bärbel_Bohley.jpg
✓ Downloaded: images/Eberhard_Bosslet.jpg
✓ Downloaded: images/Erwin_Bowien.jpg
✓ Downloaded: images/Pola_Brändle.jpg
✓ Downloaded: images/Jörg_Breu_the_Elder.jpg
✓ Downloaded: images/Jörg_Breu_the_Younger.jpg
✓ Downloaded: images/Hans_Burg